In [0]:
%sql

SELECT 
  DISTINCT
  source_table_catalog,
  source_table_schema,
  source_table_name,
  source_type,
  target_table_catalog,
  target_table_schema,
  target_table_name,
  target_table_full_name,
  target_type
  FROM system.access.table_lineage
  WHERE target_table_schema = 'nyc-taxi-demo' and target_table_catalog ='workspace'

In [0]:
%sql
SELECT *
  FROM system.access.column_lineage;

In [0]:
%sql
SELECT 
  DISTINCT
  source_table_catalog,
  source_table_schema,
  source_table_name,
  source_type,
  source_column_name,
  target_table_catalog,
  target_table_schema,
  target_table_name,
  target_type,
  target_column_name
  FROM system.access.column_lineage
  WHERE target_type is NOT NULL
    AND target_table_schema = 'nyc-taxi-demo'
    AND target_table_catalog = 'workspace'
    

### Table Level Lineage

In [0]:
%python
import requests

url = "https://dbc-2194f007-8553.cloud.databricks.com/api/2.0/lineage-tracking/table-lineage"
payload = {
    "table_name": "workspace.nyc-taxi-demo.flagged_rides", # table name
    "column_name": "zip",  # column name to find
    "include_entity_lineage": False
}
pat = ""  # PAT token - khurram.mehmood@infosys.com

response = requests.get(
    url,
    json=payload,
    headers={
        "Authorization": f"Bearer {pat}",
        "Content-Type": "application/json"
    }
)
display(response.json())

###Column level lineage - recursive / multihop

In [0]:
import requests
import json
import pprint

def get_full_lineage(
    table_name: str,
    column_name: str,
    pat: str,
    base_url: str,
    direction: str,
    lineage_list: list,
    visited_columns: set = None
):
    """
    Recursively fetches the full lineage and appends the results to a list.
    
    Args:
        table_name: The table name (e.g., "catalog.schema.table").
        column_name: The column name.
        pat: The Databricks personal access token.
        base_url: The base URL of the Databricks workspace.
        direction: "upstream" or "downstream".
        lineage_list: A list to append the lineage relationships to.
        visited_columns: A set to track visited columns and prevent loops.
    """
    if visited_columns is None:
        visited_columns = set()

    # Create a unique identifier for the column to track visited nodes
    column_id = f"{table_name}.{column_name}"

    # KM - Added to avoid forever loops
    if column_id in visited_columns:
        print("Loop detected, skipping lineage for {column_id}")
        return
    
    visited_columns.add(column_id)

    url = f"{base_url}/api/2.0/lineage-tracking/column-lineage"
    payload = {
        "table_name": table_name,
        "column_name": column_name,
        "include_entity_lineage": True
    }

    try:
        response = requests.get(
            url,
            json=payload,
            headers={
                "Authorization": f"Bearer {pat}",
                "Content-Type": "application/json"
            }
        )
        # Raises an HTTPError if the HTTP request returned an unsuccessful status code
        response.raise_for_status()

        lineage_data = response.json()

    except requests.exceptions.RequestException as e:
        print(f"Error fetching lineage for {column_id}: {e}")
        return

    # Process and append lineage data
    if direction == "upstream" and "upstream_cols" in lineage_data:
        for col in lineage_data["upstream_cols"]:
            upstream_col_name = f"{col['catalog_name']}.{col['schema_name']}.{col['table_name']}.{col['name']}"
            lineage_list.append({
                "source": upstream_col_name,
                "target": column_id,
                "relationship": "upstream"
            })
            # Recursive call for the new upstream column
            get_full_lineage(
                f"{col['catalog_name']}.{col['schema_name']}.{col['table_name']}",
                col['name'],
                pat,
                base_url,
                "upstream",
                lineage_list,
                visited_columns
            )
    elif direction == "downstream" and "downstream_cols" in lineage_data:
        for col in lineage_data["downstream_cols"]:
            downstream_col_name = f"{col['catalog_name']}.{col['schema_name']}.{col['table_name']}.{col['name']}"
            lineage_list.append({
                "source": column_id,
                "target": downstream_col_name,
                "relationship": "downstream"
            })
            # Recursive call for the new downstream column
            get_full_lineage(
                f"{col['catalog_name']}.{col['schema_name']}.{col['table_name']}",
                col['name'],
                pat,
                base_url,
                "downstream",
                lineage_list,
                visited_columns
            )

# --- Configuration ---
# Replace with your specific details
base_url = "https://dbc-2194f007-8553.cloud.databricks.com"
pat = ""
start_table = "workspace.nyc-taxi-demo.flagged_rides"
start_column = "fare_amount"

# --- Main Logic ---
# Lists to store the lineage relationships
upstream_lineage_list = []
downstream_lineage_list = []

# Fetch upstream and downstream lineage
get_full_lineage(start_table, start_column, pat, base_url, "upstream", upstream_lineage_list)
get_full_lineage(start_table, start_column, pat, base_url, "downstream", downstream_lineage_list)

# Consolidate all data into a single dictionary
final_output = {
    "starting_point": {
        "table": start_table,
        "column": start_column
    },
    "upstream_lineage": upstream_lineage_list,
    "downstream_lineage": downstream_lineage_list
}

# Convert the dictionary to a formatted JSON string
final_json_output = json.dumps(final_output, indent=4)

# Print the final JSON output
print("########### Final JSON Output ########")
print(final_json_output)

# pretty print
#print("########### Pretty Print Output ########")
#pprint.pprint(final_output , indent=4, depth=3)
